<a href="https://colab.research.google.com/github/bibidhSubedi0/NepalLawFT/blob/main/notebooks/Eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CivicLens — Fine-tune Evaluation Notebook
Evaluates a QLoRA-fine-tuned Llama-3.2-3B-Instruct model against the base model
on a held-out Nepali legal Q&A dataset.

**Metrics:** ROUGE-L, BLEU (character-level), Semantic Similarity, LLM-as-Judge (Groq)

**Models:**
- Base: `meta-llama/Llama-3.2-3B-Instruct`
- Fine-tuned: Llama-3.2-3B + LoRA adapter (r=16, alpha=32), trained on CivicLens dataset

In [ ]:
%%capture
# Pin all versions explicitly to avoid silent breakage on Colab runtime updates.
# bitsandbytes 0.45.3 is the first release with a cu128 binary (torch 2.10 / CUDA 12.8).
# peft 0.18.1 matches the version used during training (adapter_config.json: peft_version).
!pip install bitsandbytes==0.45.3
!pip install triton==3.6.0
!pip install transformers==4.46.3
!pip install peft==0.18.1
!pip install accelerate==0.34.2
!pip install sentence-transformers==3.1.1
!pip install rouge-score==0.1.2
!pip install nltk==3.9.1
!pip install groq==0.13.0
!pip install httpx==0.28.1

In [ ]:
# Get yours at: https://huggingface.co/settings/tokens
# Accept the Llama 3.2 license at: https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct
HF_TOKEN = "hf_..."

GROQ_API_KEY = "gsk_..."

In [ ]:
import os

BASE_MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

# LoRA adapter: Upload checkpoint to Colab runtime or change path accordingly.
ADAPTER_PATH  = "/content/"

TEST_FILE     = "/content/test.jsonl"

EVAL_ALL          = False
QUICK_SAMPLE_SIZE = 50

# Generation config : greedy decoding for deterministic, reproducible outputs.
MAX_NEW_TOKENS = 256
TEMPERATURE    = 0.1

GROQ_MODEL   = "llama-3.3-70b-versatile"
JUDGE_SAMPLE = 50

# Cached inference results : if these files exist, inference is skipped entirely.
FT_RESULTS_PATH   = "/content/ft_results.json"
BASE_RESULTS_PATH = "/content/base_results.json"

OUTPUT_DIR = "/content/eval_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
import json
import time
import random
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

import nltk
nltk.download("punkt",     quiet=True)
nltk.download("punkt_tab", quiet=True)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer, util
from groq import Groq
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

print(f"PyTorch : {torch.__version__}")
print(f"Device  : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
assert os.path.exists(TEST_FILE),   f"Test file not found: {TEST_FILE}"

# Adapter check is only needed if we're going to run inference
inference_needed = not (
    os.path.exists(FT_RESULTS_PATH) and os.path.exists(BASE_RESULTS_PATH)
)

if inference_needed:
    assert os.path.exists(ADAPTER_PATH), f"Adapter path not found: {ADAPTER_PATH}"
    print(f"Adapter contents ({ADAPTER_PATH}):")
    for f in sorted(os.listdir(ADAPTER_PATH)):
        size = os.path.getsize(os.path.join(ADAPTER_PATH, f))
        print(f"  {f:<40} {size / 1e6:.2f} MB")
else:
    print("Cached inference results found, model loading and inference will be skipped.")

print(f"\nTest file : {TEST_FILE}")

In [ ]:
def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

raw = load_jsonl(TEST_FILE)

eval_data = [
    {
        "question":  item["question"].strip(),
        "reference": item["answer"].strip(),
        "source":    item.get("source", ""),
    }
    for item in raw
    if item.get("question") and item.get("answer")
]

print(f"Total valid samples : {len(eval_data)}")

if not EVAL_ALL:
    random.seed(42)
    eval_data = random.sample(eval_data, min(QUICK_SAMPLE_SIZE, len(eval_data)))
    print(f"Sampled             : {len(eval_data)}")

print(f"\nSample record:")
print(f"  Q      : {eval_data[0]['question'][:120]}")
print(f"  Answer : {eval_data[0]['reference'][:120]}")
print(f"  Source : {eval_data[0]['source']}")

In [ ]:
if inference_needed:
    # 4-bit quantization config — matches training setup exactly.
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    # Tokenizer is shared between base and fine-tuned — LoRA doesn't touch it.
    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(
        BASE_MODEL_ID,
        token=HF_TOKEN,
        padding_side="left",
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("Loading base model (4-bit)...")
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        token=HF_TOKEN,
    )
    base_model.eval()
    print(f"Base model loaded — {sum(p.numel() for p in base_model.parameters()) / 1e9:.2f}B params")

    # PeftModel wraps the base model in-place — both models share the same
    # base weights in memory, only the adapter tensors are added on top.
    print("Loading fine-tuned model (base + LoRA adapter)...")
    ft_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
    ft_model.eval()
    trainable = sum(p.numel() for p in ft_model.parameters() if p.requires_grad)
    print(f"Fine-tuned model loaded — {trainable / 1e6:.2f}M trainable adapter params")
else:
    print("Skipping model load — using cached inference results.")

In [ ]:
# The system prompt must exactly match what was used during training.
# The model was fine-tuned to follow this persona and citation behavior.
SYSTEM_PROMPT = (
    "You are CivicLens, a legal assistant specialized in Nepal's laws, "
    "constitution, and governance documents. Answer questions accurately, "
    "cite your sources, and respond in the same language as the question. "
    "If you don't know, say so."
)

def build_prompt(question: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    # apply_chat_template reconstructs the exact token format used during training
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

@torch.inference_mode()
def generate(model, question: str) -> str:
    prompt = build_prompt(question)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    # Slice off the input tokens — decode only the generated portion
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


if inference_needed:
    print(f"Running inference on {len(eval_data)} samples (base + fine-tuned)...")
    records = []

    for i, item in enumerate(tqdm(eval_data, desc="Inference")):
        records.append({
            "question":  item["question"],
            "reference": item["reference"],
            "source":    item["source"],
            "base_pred": generate(base_model, item["question"]),
            "ft_pred":   generate(ft_model,   item["question"]),
        })

        # Checkpoint every 25 rows — inference is slow and Colab can disconnect
        if (i + 1) % 25 == 0:
            pd.DataFrame(records).to_csv(f"{OUTPUT_DIR}/predictions_checkpoint.csv", index=False)

    df = pd.DataFrame(records)
    df.to_csv(f"{OUTPUT_DIR}/predictions_full.csv", index=False)
    print(f"Inference complete.")

else:
    # Load from cached results — skips the ~20 min inference step
    print("Loading predictions from cached results...")
    with open(FT_RESULTS_PATH)   as f: ft_results   = json.load(f)
    with open(BASE_RESULTS_PATH) as f: base_results = json.load(f)

    df = pd.DataFrame([
        {
            "question":  ft["question"],
            "reference": ft["answer"],
            "source":    ft["source"],
            "base_pred": base["predicted"],
            "ft_pred":   ft["predicted"],
        }
        for ft, base in zip(ft_results, base_results)
    ])
    print(f"Loaded {len(df)} samples from cache.")

In [ ]:
rouge  = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
smooth = SmoothingFunction().method1

def rouge_l(reference: str, prediction: str) -> float:
    return rouge.score(reference, prediction)["rougeL"].fmeasure

def bleu(reference: str, prediction: str) -> float:
    # Character-level tokenization handles Nepali script correctly —
    # word-level tokenization would break on non-whitespace-delimited text.
    ref_chars  = list(reference)
    pred_chars = list(prediction)
    if not pred_chars:
        return 0.0
    return sentence_bleu(
        [ref_chars], pred_chars,
        smoothing_function=smooth,
        weights=(0.5, 0.5),
    )

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Lexical metrics"):
    df.at[idx, "base_rouge_l"] = rouge_l(row["reference"], row["base_pred"])
    df.at[idx, "ft_rouge_l"]   = rouge_l(row["reference"], row["ft_pred"])
    df.at[idx, "base_bleu"]    = bleu(row["reference"],    row["base_pred"])
    df.at[idx, "ft_bleu"]      = bleu(row["reference"],    row["ft_pred"])

for metric in ["rouge_l", "bleu"]:
    b = df[f"base_{metric}"].mean()
    f = df[f"ft_{metric}"].mean()
    print(f"  {metric.upper():<12} base={b:.4f}  ft={f:.4f}  delta={((f-b)/(b+1e-9)*100):+.1f}%")

In [ ]:
# Multilingual MiniLM : supports Nepali and produces reliable cosine similarity
embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

ref_embs  = embedder.encode(df["reference"].tolist(), batch_size=64, show_progress_bar=True, convert_to_tensor=True)
base_embs = embedder.encode(df["base_pred"].tolist(), batch_size=64, show_progress_bar=True, convert_to_tensor=True)
ft_embs   = embedder.encode(df["ft_pred"].tolist(),   batch_size=64, show_progress_bar=True, convert_to_tensor=True)

df["base_semantic_sim"] = util.cos_sim(ref_embs, base_embs).diagonal().cpu().numpy()
df["ft_semantic_sim"]   = util.cos_sim(ref_embs, ft_embs).diagonal().cpu().numpy()

b = df["base_semantic_sim"].mean()
f = df["ft_semantic_sim"].mean()
print(f"  Semantic similarity   base={b:.4f}  ft={f:.4f}  delta={((f-b)/(b+1e-9)*100):+.1f}%")

In [ ]:
groq_client = Groq(api_key=GROQ_API_KEY)

JUDGE_SYSTEM = """You are an expert evaluator for a Nepali legal Q&A system.

Given a question, a reference answer, and a predicted answer, score the prediction from 1 to 5:
  5 - Completely correct, matches reference meaning with no errors
  4 - Mostly correct, minor omissions or slight wording differences
  3 - Partially correct, captures main point but missing details
  2 - Mostly wrong, only superficially related to the reference
  1 - Completely wrong or irrelevant

Return ONLY a JSON object with exactly two fields:
{"score": <integer 1-5>, "reason": "<one sentence in English>"}
Output nothing else."""

def judge(question: str, reference: str, prediction: str, retries: int = 3) -> dict:
    user_msg = (
        f"Question: {question}\n\n"
        f"Reference Answer: {reference}\n\n"
        f"Predicted Answer: {prediction}"
    )
    for attempt in range(retries):
        try:
            resp = groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[
                    {"role": "system", "content": JUDGE_SYSTEM},
                    {"role": "user",   "content": user_msg},
                ],
                temperature=0.0,
                max_tokens=120,
            )
            return json.loads(resp.choices[0].message.content.strip())
        except json.JSONDecodeError:
            import re
            match = re.search(r'"score"\s*:\s*([1-5])', resp.choices[0].message.content)
            if match:
                return {"score": int(match.group(1)), "reason": "extracted from malformed response"}
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
    return {"score": -1, "reason": "failed after retries"}


random.seed(42)
judge_indices = random.sample(range(len(df)), min(JUDGE_SAMPLE, len(df)))
judge_df = df.iloc[judge_indices].copy().reset_index(drop=True)

print(f"Running LLM-as-Judge on {len(judge_df)} samples via Groq ({GROQ_MODEL})...")

base_scores, ft_scores, base_reasons, ft_reasons = [], [], [], []

for _, row in tqdm(judge_df.iterrows(), total=len(judge_df), desc="Judging"):
    time.sleep(0.1)  # stay within Groq free tier rate limits
    b = judge(row["question"], row["reference"], row["base_pred"])
    f = judge(row["question"], row["reference"], row["ft_pred"])
    base_scores.append(b["score"]);   ft_scores.append(f["score"])
    base_reasons.append(b["reason"]); ft_reasons.append(f["reason"])

judge_df["base_judge_score"]  = base_scores
judge_df["ft_judge_score"]    = ft_scores
judge_df["base_judge_reason"] = base_reasons
judge_df["ft_judge_reason"]   = ft_reasons

valid  = judge_df[judge_df["base_judge_score"] > 0]
b_avg  = valid["base_judge_score"].mean()
f_avg  = valid["ft_judge_score"].mean()
print(f"  LLM Judge (1-5)   base={b_avg:.3f}  ft={f_avg:.3f}  delta={((f_avg-b_avg)/(b_avg+1e-9)*100):+.1f}%")
print(f"  Valid responses   {len(valid)}/{len(judge_df)}")

judge_df.to_csv(f"{OUTPUT_DIR}/judge_results.csv", index=False)

In [ ]:
df["base_judge_score"] = np.nan
df["ft_judge_score"]   = np.nan
for col in ["base_judge_score", "ft_judge_score"]:
    df.loc[judge_df.index, col] = judge_df[col].values

rows = [
    ("ROUGE-L",             "base_rouge_l",      "ft_rouge_l"),
    ("BLEU (char bigram)",  "base_bleu",         "ft_bleu"),
    ("Semantic Similarity", "base_semantic_sim", "ft_semantic_sim"),
    ("LLM Judge (1-5)",     "base_judge_score",  "ft_judge_score"),
]

summary_records = []
for label, base_col, ft_col in rows:
    b = df[base_col].dropna().mean()
    f = df[ft_col].dropna().mean()
    d = (f - b) / (b + 1e-9) * 100
    summary_records.append({
        "Metric":     label,
        "Base":       round(b, 4),
        "Fine-tuned": round(f, 4),
        "Delta (%)":  f"{d:+.1f}%",
    })

summary_df = pd.DataFrame(summary_records)

print("=" * 58)
print("  EVALUATION SUMMARY — CivicLens (Llama-3.2-3B)")
print("=" * 58)
print(summary_df.to_string(index=False))
print("=" * 58)
print(f"  Eval samples  : {len(df)}")
print(f"  Judge samples : {len(valid)}")
print(f"  Base model    : {BASE_MODEL_ID}")
print(f"  Adapter       : {ADAPTER_PATH}")
print("=" * 58)

df.to_csv(f"{OUTPUT_DIR}/full_results.csv", index=False)
summary_df.to_csv(f"{OUTPUT_DIR}/summary.csv", index=False)

with open(f"{OUTPUT_DIR}/summary.json", "w") as f:
    json.dump(
        {
            "base_model": BASE_MODEL_ID,
            "adapter":    ADAPTER_PATH,
            "n_eval":     len(df),
            "n_judge":    len(valid),
            "metrics": {
                "base":      {r["Metric"]: r["Base"]       for r in summary_records},
                "finetuned": {r["Metric"]: r["Fine-tuned"] for r in summary_records},
            },
        },
        f,
        indent=2,
        ensure_ascii=False,
        default=lambda x: float(x),
    )

print(f"Results saved to {OUTPUT_DIR}/")
print(f"  full_results.csv  — per-sample predictions and scores")
print(f"  judge_results.csv — LLM judge scores and reasoning")
print(f"  summary.csv       — aggregated metrics table")
print(f"  summary.json      — machine-readable summary for logging")